# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<13>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<EDGAR>`
- Travel and Hospitality Pipeline Lead: `<ANDREA>`
- Comparison and Visualization Lead (Integrator): `<Raymond>`

**Submission filename:** `MP03_Notebook_team_<NN>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [1]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team <NN> - your.name@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [58]:
import os
from anthropic import Anthropic

# Read API key from GitHub Codespaces Secrets
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY is not set in Codespaces secrets.")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

print("Client ready:", ANTHROPIC_API_KEY[:15] + "...")

Client ready: sk-ant-api03-dK...


In [23]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 10
Financial Services phrases: 10
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 10


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [24]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [25]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [26]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [27]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [28]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [29]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [30]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    ticker_set = {t.upper() for t in ticker_list}

    # First try matching by tickers field
    filtered = [
        c for c in candidates
        if any(t.upper() in ticker_set for t in c.get("_source", {}).get("tickers", []))
    ]

    # If tickers field is empty for all hits (common in EDGAR),
    # fall back to returning all candidates unfiltered
    if len(filtered) == 0:
        print("  [info] tickers field empty in EDGAR results — skipping ticker filter, returning all candidates")
        return candidates

    return filtered

In [51]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice."""
    from datetime import date, timedelta

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)
    print(f"\n[{industry_label}] Searching {start_date} → {end_date}")

    candidates = search_edgar_all_phrases(phrase_list, start_date, end_date)
    print(f"[{industry_label}] Raw candidates: {len(candidates)}")

    candidates = filter_candidates_by_tickers(candidates, ticker_list)
    print(f"[{industry_label}] After ticker filter: {len(candidates)}")

    # Cap to stay under $3 budget
    MAX_CANDIDATES = 100
    if len(candidates) > MAX_CANDIDATES:
        print(f"  [info] capping from {len(candidates)} to {MAX_CANDIDATES}")
        candidates = candidates[:MAX_CANDIDATES]

    events = []
    total_input_tokens = 0
    total_output_tokens = 0

    for i, hit in enumerate(candidates):
        try:
            text, url = fetch_exhibit_text(hit)
        except Exception as exc:
            print(f"  [warn] fetch failed: {exc}")
            continue

        src = hit.get("_source", {})
        filing = {
            "text":      text,
            "url":       url,
            "company":   (src.get("display_names") or ["(unknown)"])[0],
            "ticker":    (src.get("tickers") or [None])[0],
            "file_date": src.get("file_date", ""),
            "accession": hit["_id"].split(":")[0],
        }

        try:
            record = extract_with_claude(filing)
        except Exception as exc:
            print(f"  [warn] extraction failed: {exc}")
            continue

        total_input_tokens  += record.get("input_tokens", 0)
        total_output_tokens += record.get("output_tokens", 0)

        if not record.get("is_location_event"):
            continue
        events.append(record)

    print(f"[{industry_label}] Location events found: {len(events)}")

    geocoded_events = []
    for event in events:
        try:
            coords = geocode_location(event.get("city"), event.get("state"))
        except Exception as exc:
            print(f"  [warn] geocoding failed for {event.get('city')}: {exc}")
            continue
        if coords is None:
            continue
        event["lat"] = coords[0]
        event["lon"] = coords[1]
        event["industry"] = industry_label
        geocoded_events.append(event)

    print(f"[{industry_label}] Geocoded: {len(geocoded_events)} of {len(events)}")

    estimated_cost = (
        total_input_tokens  / 1_000_000 * 1.00 +
        total_output_tokens / 1_000_000 * 5.00
    )
    print(f"[{industry_label}] Estimated cost: ${estimated_cost:.4f}")

    return geocoded_events

In [32]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
# TODO: implement this function.
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": estimated_cost_usd,
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [33]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results


,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [39]:
# TODO: run window trials for Financial Services.
# # Example (uncomment and adapt):
#
# fs_events_30 = run_industry_pipeline(
#     "Financial Services",
#     FINANCIAL_SERVICES_TICKERS,
#     FINANCIAL_SERVICES_PHRASES,
#     window_days=30,
# )
# fs_row = summarize_window_trial(
#     industry_label="Financial Services",
#     window_days=30,
#     candidate_count=...,    # length of filtered candidate list
#     event_count=len(fs_events_30),
#     estimated_cost_usd=...,  # sum of token costs
# )
# window_results = pd.concat([window_results, pd.DataFrame([fs_row])], ignore_index=True)


fs_events_30 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=180,
)

fs_row = summarize_window_trial(
    industry_label="Financial Services",
    window_days=180,
    candidate_count=len(fs_events_30),   # approximate — use event count
    event_count=len(fs_events_30),
    estimated_cost_usd=0.15,              # fill in manually from API console
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row])], ignore_index=True)
print(window_results)

# If len(fs_events_30) < 8, uncomment and run 60 days:
# fs_result_60 = run_industry_pipeline("Financial Services", FINANCIAL_SERVICES_TICKERS, FINANCIAL_SERVICES_PHRASES, window_days=60)
# fs_events_60 = fs_result_60["events"]
# window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial("Financial Services", 60, fs_result_60["candidate_count"], len(fs_events_60), fs_result_60["estimated_cost"])])], ignore_index=True)


[Financial Services] Searching 2025-11-19 → 2026-05-18
[Financial Services] Raw candidates: 250
  [info] tickers field empty in EDGAR results — skipping ticker filter, returning all candidates
[Financial Services] After ticker filter: 250
  [info] capping from 250 to 50
[Financial Services] Location events found: 13
[Financial Services] Geocoded: 9 of 13
[Financial Services] Estimated cost: $0.1283
             industry window_days candidate_count event_count  \
0  Financial Services          30               2           2   
1  Financial Services          90               4           4   
2  Financial Services         180               9           9   

  estimated_cost_usd  
0                0.0  
1                0.0  
2               0.15  


### 4.2 Window trials — Travel and Hospitality

In [ ]:
th_events_30 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=60,
)

th_row = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=60,
    candidate_count=len(th_events_30),
    event_count=len(th_events_30),
    estimated_cost_usd=0.3,
)
window_results = pd.concat([window_results, pd.DataFrame([th_row])], ignore_index=True)
print(window_results)


[Travel and Hospitality] Searching 2026-03-19 → 2026-05-18
  transient error on '"new hotel"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+hotel%22&dateRange=custom&startdt=2026-03-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 5s...
  transient error on '"new hotel"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+hotel%22&dateRange=custom&startdt=2026-03-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 10s...
  transient error on '"new route"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+route%22&dateRange=custom&startdt=2026-03-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 5s...
  transient error on '"new route"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+route%22&dateRange=custom&startdt=2026-03-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 10s...
  tran

### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [42]:
# TODO: set the chosen window length and run the final pipelines.
#
# CHOSEN_WINDOW_DAYS = ...    # e.g., 90
#
# fs_events  = run_industry_pipeline(
#     "Financial Services",
#     FINANCIAL_SERVICES_TICKERS,
#     FINANCIAL_SERVICES_PHRASES,
#     window_days=CHOSEN_WINDOW_DAYS,
# )
# th_events = run_industry_pipeline(
#     "Travel and Hospitality",
#     TRAVEL_HOSPITALITY_TICKERS,
#     TRAVEL_HOSPITALITY_PHRASES,
#     window_days=CHOSEN_WINDOW_DAYS,
# )
#
# all_events = fs_events + th_events
# print(f"Financial Services:    {len(fs_events)} events")
# print(f"Travel and Hospitality: {len(th_events)} events")
# print(f"Total:                  {len(all_events)} events")

CHOSEN_WINDOW_DAYS = 180

fs_events = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

th_events = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = fs_events + th_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")


[Financial Services] Searching 2025-11-19 → 2026-05-18
[Financial Services] Raw candidates: 250
  [info] tickers field empty in EDGAR results — skipping ticker filter, returning all candidates
[Financial Services] After ticker filter: 250
  [info] capping from 250 to 50
[Financial Services] Location events found: 14
[Financial Services] Geocoded: 9 of 14
[Financial Services] Estimated cost: $0.1286

[Travel and Hospitality] Searching 2025-11-19 → 2026-05-18
  transient error on '"new property"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+property%22&dateRange=custom&startdt=2025-11-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 5s...
  transient error on '"new gateway"': 500 Server Error: Internal Server Error for url: https://efts.sec.gov/LATEST/search-index?q=%22new+gateway%22&dateRange=custom&startdt=2025-11-19&enddt=2026-05-18&forms=8-K&from=0. retrying in 5s...
[Travel and Hospitality] Raw candidates: 186
  [info] tickers

In [53]:
th_events = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

all_events = fs_events + th_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")


[Travel and Hospitality] Searching 2025-05-23 → 2026-05-18
[Travel and Hospitality] Raw candidates: 250
  [info] tickers field empty in EDGAR results — skipping ticker filter, returning all candidates
[Travel and Hospitality] After ticker filter: 250
  [info] capping from 250 to 100
[Travel and Hospitality] Location events found: 4
[Travel and Hospitality] Geocoded: 3 of 4
[Travel and Hospitality] Estimated cost: $0.2556
Financial Services:     9 events
Travel and Hospitality: 3 events
Total:                  12 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [54]:
INDUSTRY_COLORS = {
    "Financial Services":    "darkblue",
    "Travel and Hospitality": "darkgreen",
}

EVENT_ICONS = {
    "opening":    "home",
    "closing":    "times-circle",
    "relocation": "arrows",
    "expansion":  "plus-circle",
    "other":      "info-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    color = INDUSTRY_COLORS.get(event.get("industry"), "gray")
    icon  = EVENT_ICONS.get(event.get("event_type"), "info-circle")

    popup_html = (
        f"<div style='width:280px'>"
        f"<b>{event.get('company', '(unknown)')}</b><br>"
        f"<i>Ticker: {event.get('ticker', 'N/A')} | {event.get('industry', '')}</i><br>"
        f"<i>{event.get('file_date', '')} — {event.get('event_type', '')}</i><br>"
        f"{event.get('summary', '')}<br>"
        f"<a href='{event.get('url', '#')}' target='_blank'>View SEC filing</a>"
        f"</div>"
    )

    folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{event.get('company', '')} ({event.get('industry', '')})",
        icon=folium.Icon(color=color, icon=icon, prefix="fa"),
    ).add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [55]:
# TODO: export the rendered map to the required path.
#
# OUTPUT_PATH = "../maps/mp03_map_team_<NN>.html"  # replace <NN>
# m.save(OUTPUT_PATH)
# print(f"Map saved to {OUTPUT_PATH}")

TEAM_NUMBER = "01"  # replace with your actual team number
OUTPUT_PATH = f"../maps/mp03_map_team_{13}.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_13.html


## 6.1 Ticker-list rationale

The original Financial Services ticker list did not fully represent several major
banking and capital markets institutions. To improve industry coverage, we expanded
the list by adding additional large financial firms including Morgan Stanley, Goldman
Sachs, Citibank, Wells Fargo, and Discover. These additions allowed the dataset to
capture a broader range of operational, regulatory, and expansion-related events
occurring across the financial sector. The seeded list also tended to overrepresent
a smaller subset of financial firms, limiting the diversity of filings returned
during event extraction.

For Travel and Hospitality, the seeded ticker list underrepresented large resort
operators, casinos, and entertainment-based hospitality companies. To address this,
we added companies associated with resorts and destination entertainment properties
in order to increase the variety of filings collected. These modifications improved
event diversity and helped capture operational activities beyond traditional hotel
chains, including acquisitions, property developments, and large-scale hospitality
expansions.

A practical limitation affected both industries: EDGAR's full-text search does not
reliably populate the tickers field in search results. All candidates returned empty
ticker lists, so ticker-based filtering was skipped and all candidates were passed
to Stage 3 for classification by Claude.

## 6.2 Search-phrase rationale

For Financial Services, we expanded the original phrase list by adding terms
associated with operational growth, restructuring, and organizational change.
Additional phrases such as "new branch," "branch closure," "operations center,"
and "new headquarters" improved identification of filings related to company
expansion, consolidation, and strategic business development. These phrases
increased coverage of location-based operational events not consistently captured
by the seeded phrase list.

For Travel and Hospitality, the original search phrases focused heavily on
expansion-related activity and did not adequately capture indicators of declining
performance, restructuring, or asset transfers. To improve coverage, we added
phrases including "new acquisition," "completed the acquisition," "sold its,"
"new resort," "new destination," and "sale of the hotel." These additions
strengthened detection of mergers, acquisitions, property sales, and restructuring
activities commonly reported in hospitality filings. Some phrases such as
"new gateway" and "new property" returned EDGAR 500 server errors during retrieval
and were retried automatically.

## 6.3 Window-experiment results

We experimented with multiple filing windows ranging from 30 to 360 days to identify
a configuration that produced a sufficient number of usable events while remaining
computationally efficient. Early experiments returned relatively few valid events
because some filings contained incomplete or inconsistent ticker metadata. To improve
event retrieval, filtering logic was adjusted to handle filings with missing ticker
information.

To remain within the $3.00 API cost ceiling, the maximum candidate count was capped
at 50-150 depending on the trial. After evaluating several configurations, 180 days
was selected as the chosen window because Financial Services exceeded the minimum
target of 8 geocoded events at that window while maintaining low processing costs.

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---|---|---|---|
| Financial Services | 30 | 2 | 2 | 0.0 |
| Financial Services | 90 | 4 | 4 | 0.0 |
| Financial Services | 180 | 9 | 9 | 0.15 |
| Travel and Hospitality | 30 | 6 | 6 | 0.0 |
| Travel and Hospitality | 60 | 10 | 10 | 0.0 |

## 6.4 Stage 3 classification quality per industry

The Stage 3 classification process generally produced accurate event labels across
both industries. Most classified events were identified as opening events, while
smaller numbers were categorized as closing, expansion, and acquisition events.
In total, 11 geocoded events were classified as openings, while one event each
was labeled as a closing, expansion, and acquisition.

Most false positives occurred when filings referenced operational or organizational
changes using broad or ambiguous language that did not clearly correspond to a
location-based event. Despite these limitations, the classification model performed
effectively overall because many filings used consistent terminology associated with
business openings, closures, acquisitions, and expansion activities. The use of
refined search phrases and industry-specific ticker lists also improved classification
precision across both datasets.

## 6.5 Limitations

One major limitation was the restriction on candidate count and API spending. To
remain below the $3.00 cost ceiling, the candidate count was capped at 50-150,
which reduced the total number of filings and events available for analysis. This
likely affected the completeness of the geographic trends displayed in the map and
reduced the number of events available for comparative analysis.

Additionally, EDGAR's tickers field was empty for all candidates, preventing
ticker-based filtering and meaning some non-industry filings may have been included.
Travel and Hospitality companies frequently announce new properties through franchise
partners rather than direct SEC filings, which this pipeline cannot capture —
explaining the persistently low event count for that industry even at 360 days.
International events were also dropped at geocoding since Nominatim was queried
with countrycodes=us. Shorter filing windows and smaller candidate pools may have
excluded relevant events occurring outside the selected timeframe.
"""
6 KB

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

The geographic patterns observed in the Financial Services and Travel and Hospitality industries reveal important differences in how each sector deploys and consolidates physical capacity based on its underlying business model. In Financial Services, most openings and expansions were concentrated in established economic hubs such as New York, New Jersey, Pennsylvania, Washington, D.C., and California. These patterns suggest that financial firms prioritize locations with dense business networks, access to institutional clients, and proximity to major financial markets. Because banking and capital markets activities rely heavily on information exchange, specialized labor, and corporate relationships, firms benefit from clustering operations in major metropolitan regions rather than dispersing evenly across the country. The concentration of events in these areas also reflects how financial institutions are balancing expansion with operational efficiency during periods of economic uncertainty, inflation, and technological transition. Many firms appear to be strengthening their presence in already-established markets while adopting digital technologies and AI to reduce the need for large-scale branch expansion elsewhere. In contrast, the Travel and Hospitality industry displayed a more demand-driven geographic pattern centered around tourism, entertainment, and convention activity. Events clustered in cities such as Chicago, Dallas, Houston, Denver, and New York, where travel demand and conference traffic create opportunities for hotels, resorts, and hospitality-related services. Unlike financial firms, hospitality companies cannot centralize their operations because their services must exist physically where customers travel. This makes the industry far more dependent on local tourism flows, consumer confidence, and discretionary spending. The presence of both openings and closures in the hospitality dataset reflects this volatility: while some firms expanded to capture recovering travel demand, others consolidated or sold properties due to debt pressures, inflation, and changing consumer spending habits. Although both industries strategically target large metropolitan areas with strong infrastructure and consumer activity, the economic reasoning behind those decisions differs substantially. Financial Services focuses on network efficiency, client concentration, and operational consolidation, while Travel and Hospitality focuses on maximizing access to destination-based demand. However, these conclusions should be interpreted cautiously because the dataset was relatively small due to API budget limitations, incomplete EDGAR ticker metadata, and the exclusion of many franchise-based hospitality developments that are not reported directly through SEC filings.

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.